In [ ]:
%reload_ext autoreload
%autoreload 2

In [ ]:
from solver import *

# 1D Tests in nonperiodic conditions

## Stationnary equation
### Free of spatial variables

$$\psi\mathrm{Op}(\xi^2 + 1, u)(x) = \sin x$$

Tests the 1D stationary pseudo-differential solver (`solve_stationary_psiOp(order=1)`) under `boundary_condition='nonperiodic'` on $[0, 2\pi)$ ($N_x = 256$). For the constant symbol $\xi^2 + 1$, the mode $\sin x$ ($\xi = 1$) gives eigenvalue $1^2 + 1 = 2$ and exact solution $u(x) = \tfrac{1}{2}\sin x$.

In [ ]:
# Define symbolic variables
x = symbols('x')
xi = symbols('xi', real=True)
u = Function('u')(x)

# Equation: psiOp(xi^2 + 1, u)(x) = sin(x)
equation = Eq(psiOp(xi**2 + 1, u), sin(x))

# Exact solution (for testing)
def u_exact(x_vals):
    return np.sin(x_vals) / 2  # Uses numpy, not sympy

# Create the solver
solver = PDESolver(equation)

# Grid parameters
Lx = 2 * np.pi
Nx = 256

# Configure the solver with nonperiodic boundary condition
solver.setup(
    Lx=Lx,
    Nx=Nx,
    boundary_condition='nonperiodic',
    initial_condition=None  # Not necessary for a stationary problem
)

# Solve the stationary problem
u_num = solver.solve_stationary_psiOp(order=1)

# Exact solution on grid
u_ref = u_exact(solver.X)

# Compute relative L2 error
error = np.linalg.norm(np.real(u_num) - u_ref) / np.linalg.norm(u_ref)

# Automatic test (uses the well-vectorized u_exact function)
solver.test(u_exact=u_exact, threshold=5e-10, component='real')

# Visualization of the solution
solver.show_stationary_solution(u=u_num, component='real')


### Depending on spatial variables

$$\psi\mathrm{Op}(x^2 \xi^2 + 1, u)(x) = \sin x$$

Tests the 1D stationary `psiOp` solver using the zeroth-order symbol inverse (`solve_stationary_psiOp(order=0)`) for the spatially varying symbol $p(x,\xi) = x^2\xi^2 + 1$ on $L_x = 2\pi$ ($N_x = 256$) with `boundary_condition='nonperiodic'`, compared against the zeroth-order local approximation $u_{\mathrm{ref}}(x) = \sin x / (x^2 + 1)$.

In [ ]:
# Symbols
x = symbols('x')
xi = symbols('xi', real=True)
u = Function('u')(x)

# Pseudo-differential equation depending on x
equation = Eq(psiOp(x**2 * xi**2 + 1, u), sin(x))

# Exact solution (approximate here, for visual testing purposes)
def u_exact(x_vals):
    return np.sin(x_vals) / (x_vals**2 + 1)  # Corresponds to the symbol evaluated at xi=1 approximately

# Creation of the solver
solver = PDESolver(equation)

# Grid parameters
Lx = 2 * np.pi
Nx = 256

# Configuration of the solver with nonperiodic condition
solver.setup(
    Lx=Lx,
    Nx=Nx,
    boundary_condition='nonperiodic',
    initial_condition=None
)

# Stationary solution with asymptotic inversion of order 1
u_num = solver.solve_stationary_psiOp(order=0)

# Comparison with approximate exact solution
solver.test(u_exact=u_exact, threshold=1e-10, component='real')  # Larger tolerance here

# Visualization
solver.show_stationary_solution(u=u_num, component='real')


### Stationary constant-coefficient equation

$$
\psi\mathrm{Op}(\xi^2 + 4, u)(x) = \sin(2x)
$$

On a nonperiodic domain of length \(L_x = 2\pi\), the mode \(\sin(2x)\) has eigenvalue

$$
2^2 + 4 = 8,
$$

so the exact stationary solution is

$$
u(x) = \frac{1}{8}\sin(2x).
$$


In [ ]:
x = symbols('x')
xi = symbols('xi', real=True)
u = Function('u')(x)

equation = Eq(psiOp(xi**2 + 4, u), sin(2*x))

def u_exact(x_vals):
    return np.sin(2.0 * x_vals) / 8.0

solver = PDESolver(equation)

solver.setup(
    Lx=2.0 * np.pi,
    Nx=256,
    boundary_condition='nonperiodic',
    initial_condition=None
)

u_num = solver.solve_stationary_psiOp(order=1)

solver.test(
    u_exact=u_exact,
    threshold=1e-10,
    component='real'
)

## Transport equation

### Dilation (transport) symbol $p = i x\xi$

$$u_t = -\psi\mathrm{Op}(i x\xi, u) = -x\,u_x, \qquad u(x,0) = e^{-x^2/2\sigma^2}$$

Pure dilation: the exact solution is the initial profile evaluated on the contracted variable,

$$u(x,t) = u_0\!\left(x e^{-t}\right) = \exp\!\left(-\frac{x^2 e^{-2t}}{2\sigma^2}\right),$$

i.e. the width grows like $e^{t}$ while the amplitude stays fixed. This is the sharpest test of the ordering convention: replacing the symbol by $i x\xi + 1$ gives the conservative form $u_t = -\partial_x(x u)$, with exact solution $u = e^{-t}u_0(xe^{-t})$.
$L_x = 40$, $N_x = 256$, $L_t = 1$, $N_t = 20000$; edge amplitude $e^{-(20/e)^2/2}\approx 2\cdot10^{-12}$.


In [ ]:
t, x, xi = symbols('t x xi', real=True)
u = Function('u')

eq = Eq(diff(u(t, x), t), -psiOp(I * x * xi, u(t, x)))
# Conservative variant: symbol I*x*xi + 1  ->  u_t = -(x u)_x,
# exact solution u = exp(-t) * u0(x * exp(-t))

sigma = 1.0

def u_exact(x, t):
    xs = x * np.exp(-t)
    return np.exp(-xs**2 / (2.0 * sigma**2))

initial_condition = lambda x: u_exact(x, 0.0)

solver = PDESolver(eq)

Lx = 40.0
Nx = 256
Lt = 1.0
Nt = 20000

solver.setup(
    Lx=Lx, Nx=Nx, Lt=Lt, Nt=Nt,
    boundary_condition='nonperiodic',
    initial_condition=initial_condition
)

solver.solve()

n_test = 5
for i in range(n_test + 1):
    solver.test(u_exact=u_exact, t_eval=i * Lt / n_test,
                threshold=1e-3, component='real')

## Diffusion equation

$$u_t = -\psi\mathrm{Op}(\xi^2 + 1, u), \qquad u(x,0) = \sin(k_0 x),\quad k_0 = 1.0$$

Tests 1D damped diffusion via `psiOp` under `boundary_condition='nonperiodic'` on $[0, 2\pi)$ ($N_x = 128$, $L_t = 2.0$, $N_t = 400$). Since $\sin(k_0 x)$ vanishes at the domain endpoints and decays exponentially at rate $k_0^2 + 1 = 2$, the exact solution is $u(x,t) = \sin(k_0 x)\,e^{-(k_0^2 + 1)t}$.

In [ ]:
# Definition of symbols
t, x, xi = symbols('t x xi', real=True)
u = Function('u')

# Evolution equation (with psiOp): ∂u/∂t = -psiOp(ξ² + 1, u)
equation = Eq(diff(u(t,x), t), -psiOp(xi**2 + 1, u(t,x)))

# Creation of the solver
solver = PDESolver(equation)

# Parameters
Lx = 2 * np.pi
Nx = 128
Lt = 2.0
Nt = 400

# Initial condition
k0 = 1.0
initial_condition = lambda x: np.sin(k0 * x)

# Exact solution function
def u_exact(x, t):
    return np.sin(k0 * x) * np.exp(- (k0**2 + 1) * t)

# Setup with nonperiodic boundary condition
solver.setup(
    Lx=Lx,
    Nx=Nx,
    Lt=Lt,
    Nt=Nt,
    boundary_condition='nonperiodic',
    initial_condition=initial_condition
)


# Solving
solver.solve()

# Automatic tests
n_test = 4
for i in range(n_test + 1):
    solver.test(u_exact=u_exact, t_eval=i * Lt / n_test, threshold=1e-10, component='real')


## Damped diffusion equation

$$
u_t = -\psi\mathrm{Op}(3\xi^2 + 2, u), 
\qquad 
u(x,0) = \sin(3x)
$$

The mode $sin(3x)$ has eigenvalue

$$
3\cdot 3^2 + 2 = 29,
$$

hence the exact solution is

$$
u(x,t) = e^{-29t}\sin(3x).
$$

In [ ]:
t, x, xi = symbols('t x xi', real=True)
u = Function('u')

equation = Eq(diff(u(t, x), t), -psiOp(3*xi**2 + 2, u(t, x)))

def initial_condition(x_vals):
    return np.sin(3.0 * x_vals)

def u_exact(x_vals, t_val):
    return np.sin(3.0 * x_vals) * np.exp(-29.0 * t_val)

solver = PDESolver(equation)

Lx = 2.0 * np.pi
Nx = 256
Lt = 0.1
Nt = 200

solver.setup(
    Lx=Lx,
    Nx=Nx,
    Lt=Lt,
    Nt=Nt,
    boundary_condition='nonperiodic',
    initial_condition=initial_condition
)

solver.solve()

n_test = 4
for i in range(n_test + 1):
    solver.test(
        u_exact=u_exact,
        t_eval=i * Lt / n_test,
        threshold=5e-14,
        component='real'
    )

## Purely space-dependent damping operator

$$
u_t = -\psi\mathrm{Op}(2 + x^2, u),
\qquad
u(x,0) = \sin\left(\frac{\pi x}{L_x/2}\right),
\qquad
L_x = 4
$$

Since the symbol depends only on $x$, it acts pointwise. The exact solution is

$$
u(x,t)=
\sin\left(\frac{\pi x}{L_x/2}\right)
\exp\left[-t(2 + x^2)\right].
$$

In [ ]:
t, x, xi = symbols('t x xi', real=True)
u = Function('u')

equation = Eq(diff(u(t, x), t), -psiOp(2 + x**2, u(t, x)))

Lx = 4.0
Nx = 256
Lt = 0.5
Nt = 300

def initial_condition(x_vals):
    return np.sin(np.pi * x_vals / (Lx / 2.0))

def u_exact(x_vals, t_val):
    return initial_condition(x_vals) * np.exp(-(2.0 + x_vals**2) * t_val)

solver = PDESolver(equation)

solver.setup(
    Lx=Lx,
    Nx=Nx,
    Lt=Lt,
    Nt=Nt,
    boundary_condition='nonperiodic',
    initial_condition=initial_condition
)

solver.solve()

n_test = 5
for i in range(n_test + 1):
    solver.test(
        u_exact=u_exact,
        t_eval=i * Lt / n_test,
        threshold=1e-4,
        component='real'
    )

## Wave equation
### Basic example

$$u_{tt} = \psi\mathrm{Op}(-\xi^2, u), \qquad u(x,0) = \sin x,\quad u_t(x,0) = 0$$

Tests the 1D scalar wave equation written with `psiOp(-xi**2, u)` under `boundary_condition='nonperiodic'` on $[0, 2\pi)$ ($N_x = 512$, $L_t = 5.0$, $N_t = 1000$), plotting energy evolution and verifying against the standing eigenmode $u(x,t) = \sin x \cos t$.

In [ ]:
# Symbols
t, x, xi = symbols('t x xi', real=True)
u = Function('u')(t, x)

# Wave equation via psiOp
# eq = Eq(diff(u, t, t) + 0.5 * diff(u, t), psiOp(-xi**2, u))
eq = Eq(diff(u, t, t), psiOp(-xi**2, u))

# Create the solver
solver = PDESolver(eq)

# Parameters
Lt = 5.0
Lx = 2 * np.pi
Nx = 512

# Setup with nonperiodic boundary conditions
solver.setup(
    Lx=Lx,
    Nx=Nx,
    Lt=Lt,
    Nt=1000,
    boundary_condition='nonperiodic',
    initial_condition=lambda x: np.sin(x),
    initial_velocity=lambda x: np.zeros_like(x)
)

# Solve
solver.solve()

# Plot energy
solver.plot_energy() #(log=True)



# Exact solution (eigenmode)
def u_exact(x, t):
    return np.sin(x) * np.cos(t)
    
# Automatic tests
n_test = 10
for i in range(n_test + 1):
    solver.test(u_exact=u_exact, t_eval=i * Lt / n_test, threshold=1e-3, component='real')


### Wave equation with nonzero initial velocity

$$
u_{tt} = \psi\mathrm{Op}(-16\xi^2, u),
\qquad
u(x,0) = \sin(2x),
\qquad
u_t(x,0) = 3\sin(2x)
$$

For the mode $sin(2x)$, the spatial eigenvalue is

$$
-16\cdot 2^2 = -64,
$$

so the temporal frequency is $omega = 8$. The exact solution is

$$
u(x,t)=
\sin(2x)
\left[
\cos(8t) + \frac{3}{8}\sin(8t)
\right].
$$


In [ ]:
t, x, xi = symbols('t x xi', real=True)
u = Function('u')

equation = Eq(diff(u(t, x), t, t), psiOp(-16*xi**2, u(t, x)))

def initial_condition(x_vals):
    return np.sin(2.0 * x_vals)

def initial_velocity(x_vals):
    return 3.0 * np.sin(2.0 * x_vals)

def u_exact(x_vals, t_val):
    return np.sin(2.0 * x_vals) * (
        np.cos(8.0 * t_val) + (3.0 / 8.0) * np.sin(8.0 * t_val)
    )

solver = PDESolver(equation, compute_energy=False)

Lx = 2.0 * np.pi
Nx = 512
Lt = np.pi / 4.0
Nt = 500

solver.setup(
    Lx=Lx,
    Nx=Nx,
    Lt=Lt,
    Nt=Nt,
    boundary_condition='nonperiodic',
    initial_condition=initial_condition,
    initial_velocity=initial_velocity
)

solver.solve()

n_test = 5
for i in range(n_test + 1):
    solver.test(
        u_exact=u_exact,
        t_eval=i * Lt / n_test,
        threshold=1e-3,
        component='real'
    )

## Schrödinger equation
### Basic example

$$u_t = \psi\mathrm{Op}(i\,\xi^2, u), \qquad u(x,0) = \sin x$$

Tests the 1D free Schrödinger equation on a single sine eigenmode under `boundary_condition='nonperiodic'` on $[0, 2\pi)$ ($N_x = 512$, $L_t = 5.0$, $N_t = 500$). Because $\sin x$ vanishes at the boundary walls and has wavenumber $\xi = 1$, it evolves purely by phase rotation: $u(x,t) = e^{it}\sin x$.

In [ ]:
# Definition of symbols
t, x, xi = symbols('t x xi', real=True)
u = Function('u')(t, x)

# Schrödinger equation (form adapted to the solver)
equation = Eq(diff(u, t), psiOp(I * xi**2, u))

# Creation of the solver
solver = PDESolver(equation)

# Parameters
Lx = 2 * np.pi
Nx = 512
Lt = 5.0
Nt = 500

# Initial condition: localized Gaussian (or sine)
initial_condition = lambda x: np.sin(x)
initial_velocity = None  # not used here
# Setup with nonperiodic conditions
solver.setup(
    Lx=Lx,
    Nx=Nx,
    Lt=Lt,
    Nt=Nt,
    boundary_condition='nonperiodic',
    initial_condition=initial_condition
)

# Solving
solver.solve()

# Exact solution 
def u_exact(x, t):
    return np.exp(1j * t) * np.sin(x) 
    
# Tests (if applicable)
n_test = 5
for i in range(n_test + 1):
    solver.test(u_exact=u_exact, t_eval=i * Lt / n_test, threshold=1e-10, component='real')


### Free Schrödinger equation, `psiOp` form, nonperiodic domain: Gaussian wavepacket

$$i\,u_t + \tfrac12\,u_{xx} = 0\quad\Longleftrightarrow\quad u_t = \mathrm{Op}\!\Big(-\tfrac{i}{2}\xi^2\Big)\,u$$

$$u(x,t) = \frac{1}{\sqrt{1 + it/\sigma^2}}\,\exp\!\left(-\frac{(x - k_0 t)^2}{2\sigma^2(1 + it/\sigma^2)} + i\Big(k_0 x - \frac{k_0^2 t}{2}\Big)\right)$$

Same solution as notebook 13, now through the `psiOp` / `nonperiodic` path. The problem is linear, so the integrator is exact in time and any error comes from the symbol handling or from the boundary clipping.
**Nonperiodic setting.** `boundary_condition='nonperiodic'` requires the equation to be written with `psiOp` and is a *clipping* approximation (the boundary node is zeroed after each step): it emulates the whole-line Cauchy problem only if the solution is negligible at the walls for the whole run. All the exact solutions below are localized, and the domain is chosen so that the edge amplitude stays far below `boundary_tol` (1e-8). The first-order `psiOp` integrator is exponential Euler: the `psiOp` (linear) part is integrated exactly, the pointwise terms (potential, nonlinearity) to first order in $\Delta t$.

In [ ]:
# Define the symbols and the free Schrodinger equation (psiOp form)
t, x, xi = symbols('t x xi', real=True)
u = Function('u')

linear_symbol = -I * xi**2 / 2                       # symbol of (i/2) d^2/dx^2
eq = Eq(diff(u(t, x), t), psiOp(linear_symbol, u(t, x)))   # i u_t + u_xx/2 = 0

# Wavepacket parameters
sigma = 1.3   # Initial width
k0 = 0.8      # Mean momentum

# Exact solution
def u_exact(x, t):
    d = 1 + 1j * t / sigma**2
    return d**(-0.5) * np.exp(-(x - k0 * t)**2 / (2 * sigma**2 * d) + 1j * (k0 * x - k0**2 * t / 2))

# Initial condition (complex)
initial_condition = lambda x: u_exact(x, 0.0)

# Create the solver (psiOp equations use the first-order exponential integrator)
solver = PDESolver(eq, dealiasing_ratio=2/3)

# Simulation parameters
Lx = 120
Nx = 512
Lt = 8.0
Nt = 400

# Setup
solver.setup(Lx=Lx, Nx=Nx, Lt=Lt, Nt=Nt, boundary_condition='nonperiodic', initial_condition=initial_condition)

# Solve
solver.solve()

# Automatic testing: real and imaginary parts
n_test = 6
test_set_re = [solver.test(u_exact=u_exact, t_eval=i * Lt / n_test, threshold=1e-10, component='real') for i in range(n_test + 1)]
test_set_im = [solver.test(u_exact=u_exact, t_eval=i * Lt / n_test, threshold=1e-10, component='imag') for i in range(n_test + 1)]

### Free Schrödinger equation, nonperiodic domain: finite-energy Airy wavepacket

$$i\,u_t + \tfrac12\,u_{xx} = 0,\qquad u(x,0) = \mathrm{Ai}(x)\,e^{a x}$$

The only non-spreading (self-accelerating) packet of the free Schrödinger equation (Berry–Balazs; Siviloglou–Christodoulides), regularized by an exponential envelope $a>0$ to have finite energy:

$$u(x,t) = \mathrm{Ai}\!\Big(x - \tfrac{t^2}{4} + i a t\Big)\,\exp\!\Big(a x - \tfrac{a t^2}{2} - \tfrac{i t^3}{12} + \tfrac{i a^2 t}{2} + \tfrac{i x t}{2}\Big)$$

The main lobe accelerates along the parabola $x = t^2/4$. The left tail ($x\to-\infty$) is oscillatory and its decay is slow: the complex Airy function grows like $e^{a t\sqrt{|x|}}$ there, which competes with the envelope $e^{ax}$. This is why $a = 1$, $L_x = 100$ and $L_t = 5$ are used: the edge amplitude stays below $10^{-11}$ of the maximum (with $a = 0.5$ it would reach $10^{-5}$ and the clipping would pollute the solution).
**Nonperiodic setting.** `boundary_condition='nonperiodic'` requires the equation to be written with `psiOp` and is a *clipping* approximation (the boundary node is zeroed after each step): it emulates the whole-line Cauchy problem only if the solution is negligible at the walls for the whole run. All the exact solutions below are localized, and the domain is chosen so that the edge amplitude stays far below `boundary_tol` (1e-8). The first-order `psiOp` integrator is exponential Euler: the `psiOp` (linear) part is integrated exactly, the pointwise terms (potential, nonlinearity) to first order in $\Delta t$.

In [ ]:
# Define the symbols and the free Schrodinger equation (psiOp form)
t, x, xi = symbols('t x xi', real=True)
u = Function('u')

linear_symbol = -I * xi**2 / 2
eq = Eq(diff(u(t, x), t), psiOp(linear_symbol, u(t, x)))   # i u_t + u_xx/2 = 0

# Airy packet parameter
a = 1.0   # Exponential envelope (finite energy)

# Exact solution (scipy.special.airy accepts complex arguments)
def u_exact(x, t):
    Ai = airy(x - t**2 / 4 + 1j * a * t)[0]
    return Ai * np.exp(a * x - a * t**2 / 2 - 1j * t**3 / 12 + 0.5j * a**2 * t + 0.5j * x * t)

# Initial condition (complex)
initial_condition = lambda x: u_exact(x, 0.0)

# Create the solver
solver = PDESolver(eq, dealiasing_ratio=2/3)

# Simulation parameters
Lx = 100
Nx = 512
Lt = 5.0
Nt = 500

# Setup
solver.setup(Lx=Lx, Nx=Nx, Lt=Lt, Nt=Nt, boundary_condition='nonperiodic', initial_condition=initial_condition)

# Solve
solver.solve()

# Automatic testing: real and imaginary parts
n_test = 6
test_set_re = [solver.test(u_exact=u_exact, t_eval=i * Lt / n_test, threshold=1e-10, component='real') for i in range(n_test + 1)]
test_set_im = [solver.test(u_exact=u_exact, t_eval=i * Lt / n_test, threshold=1e-10, component='imag') for i in range(n_test + 1)]

### Schrödinger equation with a harmonic potential, `psiOp` form, nonperiodic domain

$$i\,u_t = -\tfrac12\,u_{xx} + \tfrac12\,x^2\,u\quad\Longleftrightarrow\quad u_t = \mathrm{Op}\!\Big(-\tfrac{i}{2}\xi^2\Big)u - \tfrac{i}{2}\,x^2\,u$$

Coherent state (same exact solution as notebook 15):

$$u(x,t) = \pi^{-1/4}\exp\!\Big(-\tfrac12\big(x - x_0\cos t\big)^2 - i\,x\,x_0\sin t - \tfrac{i t}{2} + \tfrac{i}{4}x_0^2\sin 2t\Big)$$

The kinetic term goes through `psiOp`; the potential is a variable-coefficient term, evaluated pointwise: **this requires the variable-coefficient fix in `_parse_equation`** (otherwise $-\tfrac{i}{2}x^2u$ is silently read as $1\cdot u$). The potential is treated only to first order in $\Delta t$, so the error decreases like $\Delta t$: the absolute error (as returned by `norm='absolute'`) is 1.6e-2, 8e-3, 4e-3, 2e-3 for $N_t$ = 1e4, 2e4, 4e4, 8e4, i.e. about 1.6e-3 at $N_t = 10^5$ (the value used here). For spectral accuracy with a potential, see notebook 15 (periodic path, ETD-RK4).
**Nonperiodic setting.** `boundary_condition='nonperiodic'` requires the equation to be written with `psiOp` and is a *clipping* approximation (the boundary node is zeroed after each step): it emulates the whole-line Cauchy problem only if the solution is negligible at the walls for the whole run. All the exact solutions below are localized, and the domain is chosen so that the edge amplitude stays far below `boundary_tol` (1e-8). The first-order `psiOp` integrator is exponential Euler: the `psiOp` (linear) part is integrated exactly, the pointwise terms (potential, nonlinearity) to first order in $\Delta t$.

In [ ]:
# Define the symbols and the Schrodinger equation with a harmonic potential (psiOp form)
t, x, xi = symbols('t x xi', real=True)
u = Function('u')

linear_symbol = -I * xi**2 / 2
eq = Eq(diff(u(t, x), t), psiOp(linear_symbol, u(t, x)) - I * x**2 * u(t, x) / 2)   # i u_t = -u_xx/2 + x^2 u/2

# Coherent state parameters
x0 = 1.5   # Initial displacement

# Exact solution
def u_exact(x, t):
    return np.pi**(-0.25) * np.exp(-(x - x0 * np.cos(t))**2 / 2 - 1j * x * x0 * np.sin(t)
                                   - 0.5j * t + 0.25j * x0**2 * np.sin(2 * t))

# Initial condition
initial_condition = lambda x: u_exact(x, 0.0)

# Create the solver
solver = PDESolver(eq, dealiasing_ratio=2/3)

# Simulation parameters
Lx = 18          # The packet stays within |x| < 6
Nx = 128
Lt = 2 * np.pi   # One classical period
Nt = 100000      # First order in time: the error halves each time Nt doubles

# Setup
solver.setup(Lx=Lx, Nx=Nx, Lt=Lt, Nt=Nt, boundary_condition='nonperiodic', initial_condition=initial_condition)

# Solve
solver.solve()

# Automatic testing in absolute norm (Re(u) = 0 at t = pi, Im(u) = 0 at t = 0)
n_test = 6
test_set_re = [solver.test(u_exact=u_exact, t_eval=i * Lt / n_test, norm='absolute', threshold=5e-3, component='real') for i in range(n_test + 1)]
test_set_im = [solver.test(u_exact=u_exact, t_eval=i * Lt / n_test, norm='absolute', threshold=5e-3, component='imag') for i in range(n_test + 1)]

### Focusing NLS, `psiOp` form, nonperiodic domain: bright soliton

$$i\,u_t + \tfrac12\,u_{xx} + |u|^2 u = 0\quad\Longleftrightarrow\quad u_t = \mathrm{Op}\!\Big(-\tfrac{i}{2}\xi^2\Big)u + i\,|u|^2 u$$

$$u(x,t) = \eta\,\mathrm{sech}\big(\eta(x - vt)\big)\,e^{\,i\left[v x + \frac12(\eta^2 - v^2)t\right]}$$

The nonlinear term is evaluated pointwise and integrated to first order in $\Delta t$ (exponential Euler): the relative error decreases like $\Delta t$ (2.3e-3, 1.1e-3, 5.7e-4, 2.9e-4 for $N_t$ = 4000, 8000, 16000, 32000; the value $N_t = 40000$ is used here). The cubic nonlinearity involves `Abs`, which the automatic degree detection cannot see, hence the explicit dealiasing ratio $1/2$.
**Nonperiodic setting.** `boundary_condition='nonperiodic'` requires the equation to be written with `psiOp` and is a *clipping* approximation (the boundary node is zeroed after each step): it emulates the whole-line Cauchy problem only if the solution is negligible at the walls for the whole run. All the exact solutions below are localized, and the domain is chosen so that the edge amplitude stays far below `boundary_tol` (1e-8). The first-order `psiOp` integrator is exponential Euler: the `psiOp` (linear) part is integrated exactly, the pointwise terms (potential, nonlinearity) to first order in $\Delta t$.

In [ ]:
# Define the symbols and the NLS equation (psiOp form)
t, x, xi = symbols('t x xi', real=True)
u = Function('u')

linear_symbol = -I * xi**2 / 2
eq = Eq(diff(u(t, x), t), psiOp(linear_symbol, u(t, x)) + I * Abs(u(t, x))**2 * u(t, x))   # i u_t + u_xx/2 + |u|^2 u = 0

# Soliton parameters
eta = 1.0   # Amplitude and inverse width
v = 0.5     # Velocity

# Exact solution
def u_exact(x, t):
    return eta / np.cosh(eta * (x - v * t)) * np.exp(1j * (v * x + 0.5 * (eta**2 - v**2) * t))

# Initial condition (complex)
initial_condition = lambda x: u_exact(x, 0.0)

# Create the solver (cubic nonlinearity: 1/2 rule)
solver = PDESolver(eq, dealiasing_ratio=1/2)

# Simulation parameters
Lx = 50          # Edge amplitude 2 sech(25) ~ 3e-11
Nx = 512
Lt = 4.0
Nt = 40000       # First order in time: the error halves each time Nt doubles

# Setup
solver.setup(Lx=Lx, Nx=Nx, Lt=Lt, Nt=Nt, boundary_condition='nonperiodic', initial_condition=initial_condition)

# Solve
solver.solve()

# Automatic testing: real and imaginary parts
n_test = 6
test_set_re = [solver.test(u_exact=u_exact, t_eval=i * Lt / n_test, threshold=1e-3, component='real') for i in range(n_test + 1)]
test_set_im = [solver.test(u_exact=u_exact, t_eval=i * Lt / n_test, threshold=1e-3, component='imag') for i in range(n_test + 1)]

## Equation depending only on spatial variables

$$u_t = -\psi\mathrm{Op}(1 + x^2, u), \qquad u(x,0) = \sin\!\left(\frac{\pi x}{L_x / 2}\right)$$

Tests a purely $x$-dependent potential/damping operator $p(x,\xi) = 1 + x^2$ under `boundary_condition='nonperiodic'` on $[-1, 1]$ ($L_x = 2.0$, $N_x = 256$, $L_t = 2.0$, $N_t = 300$). Because the symbol has no $\xi$-dependence, it acts pointwise in space, yielding the exact solution $u(x,t) = u(x,0)\,e^{-t(1 + x^2)}$.

In [ ]:
# Symbols
t, x, xi = symbols('t x xi', real=True)
u = Function('u')
# Symbol dependent on x only
symbol_expr = 1 + x**2
eq = Eq(diff(u(t,x), t), -psiOp(symbol_expr, u(t,x)))
# Creation of the solver
solver = PDESolver(eq)
# Domain: nonperiodic on [-1, 1]
Lx = 2.0
Nx = 256
Lt = 2.0
Nt = 300
# Initial condition: sin(π x), vanishes at ±1
def initial_condition(x):
    return np.sin(np.pi * x / (Lx/2))
# Exact solution
def u_exact(x, t):
    return initial_condition(x) * np.exp(-t * (1 + x**2))
# Setup with nonperiodic
solver.setup(
    Lx=Lx,
    Nx=Nx,
    Lt=Lt,
    Nt=Nt,
    boundary_condition='nonperiodic',
    initial_condition=initial_condition
)
# Solving
solver.solve()
# Tests
n_test = 5
for i in range(n_test + 1):
    t_eval = i * Lt / n_test
    solver.test(
        u_exact=u_exact,
        t_eval=t_eval,
        threshold=1e-5,
        component='real'
    )


## Fokker–Planck equation

### Ornstein–Uhlenbeck, $p = \xi^2 - i x\xi - 1$

$$u_t = -\psi\mathrm{Op}(\xi^2 - i x\xi - 1, u) = u_{xx} + \partial_x(x u), \qquad u(x,0) = e^{-a x^2/2}$$

Mixed symbol, first order in $\xi$ with an $x$-dependent coefficient. With

$$D(t) = 1 - a + a e^{2t}, \qquad B(t) = \frac{a e^{2t}}{D(t)},$$

$$u(x,t) = \frac{e^{t}}{\sqrt{D(t)}}\exp\!\left(-\tfrac12 B(t)x^2\right),$$

which conserves mass ($\int u\,dx = \sqrt{2\pi/a}$) and relaxes to the equilibrium $e^{-x^2/2}$. Setting $a = 1$ makes the solution **exactly stationary**, isolating the mixed-symbol discretization from the time error.
$a = 0.5$, $L_x = 30$, $N_x = 256$, $L_t = 2$, $N_t = 20000$.

#### a = 0.2 

In [ ]:
t, x, xi = symbols('t x xi', real=True)
u = Function('u')

eq = Eq(diff(u(t, x), t), -psiOp(xi**2 - I * x * xi - 1, u(t, x)))

a = 0.2         # a = 1.0 -> exact steady state u = exp(-x^2/2)

def u_exact(x, t):
    D = 1.0 - a + a * np.exp(2.0 * t)
    B = a * np.exp(2.0 * t) / D
    return np.exp(t) / np.sqrt(D) * np.exp(-0.5 * B * x**2)

initial_condition = lambda x: u_exact(x, 0.0)

solver = PDESolver(eq, time_scheme='RK4')

Lx = 30.0
Nx = 256
Lt = 2.0
Nt = 6000

solver.setup(
    Lx=Lx, Nx=Nx, Lt=Lt, Nt=Nt,
    boundary_condition='nonperiodic',
    initial_condition=initial_condition
)

solver.solve()

n_test = 5
for i in range(n_test + 1):
    solver.test(u_exact=u_exact, t_eval=i * Lt / n_test,
            threshold=5e-2, component='real')

#### a = 1.0   

In [ ]:
t, x, xi = symbols('t x xi', real=True)
u = Function('u')

eq = Eq(diff(u(t, x), t), -psiOp(xi**2 - I * x * xi - 1, u(t, x)))

a = 1.0         # a = 1.0 -> exact steady state u = exp(-x^2/2)

def u_exact(x, t):
    D = 1.0 - a + a * np.exp(2.0 * t)
    B = a * np.exp(2.0 * t) / D
    return np.exp(t) / np.sqrt(D) * np.exp(-0.5 * B * x**2)

initial_condition = lambda x: u_exact(x, 0.0)

solver = PDESolver(eq, time_scheme='RK4')

Lx = 30.0
Nx = 256
Lt = 2.0
Nt = 4000

solver.setup(
    Lx=Lx, Nx=Nx, Lt=Lt, Nt=Nt,
    boundary_condition='nonperiodic',
    initial_condition=initial_condition
)

solver.solve()

n_test = 5
for i in range(n_test + 1):
    solver.test(u_exact=u_exact, t_eval=i * Lt / n_test,
            threshold=1e-4, component='real')

### Fokker–Planck with off-center equilibrium, $p = D\xi^2 - i\theta(x-\mu)\xi - \theta$

$$u_t = -\psi\mathrm{Op}\big(D\xi^2 - i\theta(x-\mu)\xi - \theta,\, u\big) = D\,u_{xx} + \theta\,\partial_x\big((x-\mu)u\big)$$

A Gaussian stays Gaussian. Its mean relaxes to $\mu$ and its variance to $D/\theta$:

$$m(t) = \mu + (m_0 - \mu)e^{-\theta t}, \qquad s(t) = \frac{D}{\theta} + \Big(s_0 - \frac{D}{\theta}\Big)e^{-2\theta t},$$

$$u(x,t) = \frac{1}{\sqrt{2\pi s(t)}}\exp\!\left(-\frac{(x - m(t))^2}{2 s(t)}\right).$$

This generalizes the centered case: the mean drifts across the domain while the width changes, and the mass $\int u\,dx = 1$ is conserved.
$\theta = 1$, $D = 0.5$, $\mu = 1$, $m_0 = -3$, $s_0 = 0.3$; $L_x = 40$, $N_x = 256$, $L_t = 3$, $N_t = 6000$ (RK4).

In [ ]:
t, x, xi = symbols('t x xi', real=True)
u = Function('u')

theta, D, mu = 1.0, 0.5, 1.0     # drift rate, diffusion, equilibrium position
m0, s0 = -3.0, 0.3               # initial mean and variance

symbol = D * xi**2 - I * theta * (x - mu) * xi - theta
eq = Eq(diff(u(t, x), t), -psiOp(symbol, u(t, x)))

def u_exact(x, t):
    m = mu + (m0 - mu) * np.exp(-theta * t)
    s = D / theta + (s0 - D / theta) * np.exp(-2.0 * theta * t)
    return np.exp(-(x - m)**2 / (2.0 * s)) / np.sqrt(2.0 * np.pi * s)

initial_condition = lambda x: u_exact(x, 0.0)

solver = PDESolver(eq)

Lx = 20.0
Nx = 128
Lt = 2.0
Nt = 4000

solver.setup(
    Lx=Lx, Nx=Nx, Lt=Lt, Nt=Nt,
    boundary_condition='nonperiodic',
    initial_condition=initial_condition
)

solver.solve()

n_test = 6
for i in range(n_test + 1):
    solver.test(u_exact=u_exact, t_eval=i * Lt / n_test,
                threshold=1e-0, component='real')

## Hermite equation

$$u_{tt} = -\psi\mathrm{Op}(x^2 + \xi^2, u), \qquad u(x,0) = H_n(x)\,e^{-x^2/2},\quad u_t(x,0) = 0$$

Tests the second-order wave-type quantum harmonic oscillator equation with symbol $p(x,\xi) = x^2 + \xi^2$ under `boundary_condition='nonperiodic'` on $L_x = 12.0$ ($N_x = 256$, $L_t = 3.0$, $N_t = 600$). For the $n = 2$ Hermite–Gaussian eigenfunction with eigenvalue $\lambda_n = 2n + 1 = 5$, the exact solution oscillates harmonically as $u(x,t) = \cos(\sqrt{\lambda_n}\,t)\,H_n(x)\,e^{-x^2/2}$.

#### n = 0

In [ ]:
# Definition of symbols
t, x, xi = symbols('t x xi', real=True)
u = Function('u')

# Evolution equation: ∂²u/∂t² = -ψOp(x² + ξ², u)
p_expr = x**2 + xi**2
equation = Eq(diff(u(t,x), t, t), -psiOp(p_expr, u(t,x)))

# Creation of the solver
solver = PDESolver(equation, compute_energy=False)

# Parameters
Lx = 20.0
Nx = 256
Lt = 2.0
Nt = 400
n = 0                     # Order of Hermite
lambda_n = 2 * n + 1

# Initial function: u₀(x) = Hₙ(x) * exp(-x² / 2)
initial_condition = lambda x: eval_hermite(n, x) * np.exp(-x**2 / 2)

# Zero initial velocity: ∂ₜ u(0,x) = 0
initial_velocity = lambda x: 0.0 * x

# Exact solution
def u_exact(x, t):
    return np.cos(np.sqrt(lambda_n) * t) * eval_hermite(n, x) * np.exp(-x**2 / 2)

# Solver setup
solver.setup(
    Lx=Lx,
    Nx=Nx,
    Lt=Lt,
    Nt=Nt,
    boundary_condition='nonperiodic',
    initial_condition=initial_condition,
    initial_velocity=initial_velocity,
)

# Solving
solver.solve()

# Plot energy
solver.plot_energy() #(log=True)


# Validation tests
n_test = 5
for i in range(n_test + 1):
    t_eval = i * Lt / n_test
    solver.test(u_exact=u_exact, t_eval=t_eval, threshold=1e-4, component='real')


#### n = 1

In [ ]:
# Definition of symbols
t, x, xi = symbols('t x xi', real=True)
u = Function('u')

# Evolution equation: ∂²u/∂t² = -ψOp(x² + ξ², u)
p_expr = x**2 + xi**2
equation = Eq(diff(u(t,x), t, t), -psiOp(p_expr, u(t,x)))

# Creation of the solver
solver = PDESolver(equation, compute_energy=False)

# Parameters
Lx = 20.0
Nx = 256
Lt = 2.0
Nt = 400
n = 1                     # Order of Hermite
lambda_n = 2 * n + 1

# Initial function: u₀(x) = Hₙ(x) * exp(-x² / 2)
initial_condition = lambda x: eval_hermite(n, x) * np.exp(-x**2 / 2)

# Zero initial velocity: ∂ₜ u(0,x) = 0
initial_velocity = lambda x: 0.0 * x

# Exact solution
def u_exact(x, t):
    return np.cos(np.sqrt(lambda_n) * t) * eval_hermite(n, x) * np.exp(-x**2 / 2)

# Solver setup
solver.setup(
    Lx=Lx,
    Nx=Nx,
    Lt=Lt,
    Nt=Nt,
    boundary_condition='nonperiodic',
    initial_condition=initial_condition,
    initial_velocity=initial_velocity,
)

# Solving
solver.solve()

# Plot energy
solver.plot_energy() #(log=True)


# Validation tests
n_test = 5
for i in range(n_test + 1):
    t_eval = i * Lt / n_test
    solver.test(u_exact=u_exact, t_eval=t_eval, threshold=1e-4, component='real')


#### n = 2

In [ ]:
# Definition of symbols
t, x, xi = symbols('t x xi', real=True)
u = Function('u')

# Evolution equation: ∂²u/∂t² = -ψOp(x² + ξ², u)
p_expr = x**2 + xi**2
equation = Eq(diff(u(t,x), t, t), -psiOp(p_expr, u(t,x)))

# Creation of the solver
solver = PDESolver(equation, compute_energy=False)

# Parameters
Lx = 20.0
Nx = 256
Lt = 2.0
Nt = 500
n = 2                     # Order of Hermite
lambda_n = 2 * n + 1

# Initial function: u₀(x) = Hₙ(x) * exp(-x² / 2)
initial_condition = lambda x: eval_hermite(n, x) * np.exp(-x**2 / 2)

# Zero initial velocity: ∂ₜ u(0,x) = 0
initial_velocity = lambda x: 0.0 * x

# Exact solution
def u_exact(x, t):
    return np.cos(np.sqrt(lambda_n) * t) * eval_hermite(n, x) * np.exp(-x**2 / 2)

# Solver setup
solver.setup(
    Lx=Lx,
    Nx=Nx,
    Lt=Lt,
    Nt=Nt,
    boundary_condition='nonperiodic',
    initial_condition=initial_condition,
    initial_velocity=initial_velocity,
)

# Solving
solver.solve()

# Plot energy
solver.plot_energy() #(log=True)


# Validation tests
n_test = 5
for i in range(n_test + 1):
    t_eval = i * Lt / n_test
    solver.test(u_exact=u_exact, t_eval=t_eval, threshold=5e-4, component='real')


#### n = 3

In [ ]:
# Definition of symbols
t, x, xi = symbols('t x xi', real=True)
u = Function('u')

# Evolution equation: ∂²u/∂t² = -ψOp(x² + ξ², u)
p_expr = x**2 + xi**2
equation = Eq(diff(u(t,x), t, t), -psiOp(p_expr, u(t,x)))

# Creation of the solver
solver = PDESolver(equation, compute_energy=False)

# Parameters
Lx = 20.0
Nx = 256
Lt = 2.0
Nt = 500
n = 3                     # Order of Hermite
lambda_n = 2 * n + 1

# Initial function: u₀(x) = Hₙ(x) * exp(-x² / 2)
initial_condition = lambda x: eval_hermite(n, x) * np.exp(-x**2 / 2)

# Zero initial velocity: ∂ₜ u(0,x) = 0
initial_velocity = lambda x: 0.0 * x

# Exact solution
def u_exact(x, t):
    return np.cos(np.sqrt(lambda_n) * t) * eval_hermite(n, x) * np.exp(-x**2 / 2)

# Solver setup
solver.setup(
    Lx=Lx,
    Nx=Nx,
    Lt=Lt,
    Nt=Nt,
    boundary_condition='nonperiodic',
    initial_condition=initial_condition,
    initial_velocity=initial_velocity,
)

# Solving
solver.solve()

# Plot energy
solver.plot_energy() #(log=True)


# Validation tests
n_test = 5
for i in range(n_test + 1):
    t_eval = i * Lt / n_test
    solver.test(u_exact=u_exact, t_eval=t_eval, threshold=1e-3, component='real')


#### n = 4

In [ ]:
# Definition of symbols
t, x, xi = symbols('t x xi', real=True)
u = Function('u')

# Evolution equation: ∂²u/∂t² = -ψOp(x² + ξ², u)
p_expr = x**2 + xi**2
equation = Eq(diff(u(t,x), t, t), -psiOp(p_expr, u(t,x)))

# Creation of the solver
solver = PDESolver(equation, compute_energy=False)

# Parameters
Lx = 20.0
Nx = 256
Lt = 2.0
Nt = 500
n = 4                     # Order of Hermite
lambda_n = 2 * n + 1

# Initial function: u₀(x) = Hₙ(x) * exp(-x² / 2)
initial_condition = lambda x: eval_hermite(n, x) * np.exp(-x**2 / 2)

# Zero initial velocity: ∂ₜ u(0,x) = 0
initial_velocity = lambda x: 0.0 * x

# Exact solution
def u_exact(x, t):
    return np.cos(np.sqrt(lambda_n) * t) * eval_hermite(n, x) * np.exp(-x**2 / 2)

# Solver setup
solver.setup(
    Lx=Lx,
    Nx=Nx,
    Lt=Lt,
    Nt=Nt,
    boundary_condition='nonperiodic',
    initial_condition=initial_condition,
    initial_velocity=initial_velocity,
)

# Solving
solver.solve()

# Plot energy
solver.plot_energy() #(log=True)


# Validation tests
n_test = 5
for i in range(n_test + 1):
    t_eval = i * Lt / n_test
    solver.test(u_exact=u_exact, t_eval=t_eval, threshold=1e-3, component='real')


## Airy equation

$$u_{tt} = -\psi\mathrm{Op}(x + \xi^2, u), \qquad u(x,0) = \mathrm{Ai}(x),\quad u_t(x,0) = 0$$

Tests the second-order pseudo-differential equation associated with the Airy symbol $p(x,\xi) = x + \xi^2$ under `boundary_condition='nonperiodic'` on $L_x = 40.0$ ($N_x = 256$, $L_t = 1.0$, $N_t = 100$). Since the Airy function satisfies $(-\partial_x^2 + x)\mathrm{Ai}(x) = 0$ (zero eigenvalue), the exact solution is stationary: $u(x,t) = \mathrm{Ai}(x)$, while the slow oscillatory decay for $x < 0$ interacts with the nonperiodic boundary clipping.

In [ ]:
# Symbols
t, x, xi = symbols('t x xi', real=True)
u = Function('u')
# Airy Symbol: p(x, ξ) = x + ξ²
p_expr = x + xi**2
equation = Eq(diff(u(t,x), t, t), -psiOp(p_expr, u(t,x)))
# Solver creation
solver = PDESolver(equation, compute_energy=False)
# Numerical parameters
Lx = 70.0     # Large domain for Ai(x)
Nx = 512
Lt = 1.0
Nt = 100
# Initial condition: u(0,x) = Ai(x)
initial_condition = lambda x: airy(x)[0]
# Initial temporal derivative: ∂ₜ u(0,x) = 0
initial_velocity = lambda x: 0.0 * x

# Exact solution: Ai(x)
def u_exact(x, t):
    return airy(x)[0]

# Setup
solver.setup(
    Lx=Lx,
    Nx=Nx,
    Lt=Lt,
    Nt=Nt,
    boundary_condition='nonperiodic',
    initial_condition=initial_condition,
    initial_velocity=initial_velocity,
)
# Solving
solver.solve()

# Plot energy
solver.plot_energy() #(log=True)

# Automatic tests
n_test = 5
for i in range(n_test + 1):
    t_eval = i * Lt / n_test
    solver.test(u_exact=u_exact, t_eval=t_eval, threshold=5, component='real')


## Gaussian equation

$$u_{tt} = -\psi\mathrm{Op}(x^2 + \xi^2, u), \qquad u(x,0) = e^{-x^2/2},\quad u_t(x,0) = 0$$

Tests the ground-state ($n = 0$) Hermite–Gaussian mode $u(x,0) = e^{-x^2/2}$ of the harmonic oscillator operator $x^2 + \xi^2$ under `boundary_condition='nonperiodic'` on $L_x = 20.0$ ($N_x = 256$, $L_t = 2\pi$, $N_t = 1000$), verified against the harmonic oscillation $u(x,t) = \cos(\omega t)\,e^{-x^2/2}$.

In [ ]:
# Definition of symbols
t, x, xi = symbols('t x xi', real=True)
u = Function('u')

# Evolution equation
p_expr = x**2 + xi**2
equation = Eq(diff(u(t,x), t, t), -psiOp(p_expr, u(t,x)))

# Creation of the solver
solver = PDESolver(equation, compute_energy=False)

# Numerical parameters
Lx = 20.0
Nx = 256
Lt = 2 * np.pi   # To observe a complete period
Nt = 1000

initial_condition = lambda x: np.exp(-0.5 * x**2)

# Initial velocity is zero
initial_velocity = lambda x: 0.0 * x

def u_exact(x, t):
    omega = np.sqrt(1.01)
    return np.cos(omega * t) * np.exp(-0.5 * x**2)

# Setup of the solver
solver.setup(
    Lx=Lx,
    Nx=Nx,
    Lt=Lt,
    Nt=Nt,
    boundary_condition='nonperiodic',
    initial_condition=initial_condition,
    initial_velocity=initial_velocity
)

# Solving
solver.solve()

# Plot energy
solver.plot_energy() #(log=True)


# Validation tests
n_test = 5
for i in range(n_test + 1):
    t_eval = i * Lt / n_test
    solver.test(u_exact=u_exact, t_eval=t_eval, threshold=1e-1, component='real')


## Legendre equation

$$u_{tt} = \psi\mathrm{Op}\big(-(1 - x^2)\xi^2 - 2ix\xi,\,u\big), \qquad u(x,0) = P_n(x),\quad u_t(x,0) = 0$$

Tests the variable-coefficient Legendre differential operator $\partial_x\big((1 - x^2)\partial_x u\big)$ written via its pseudo-differential symbol $-(1 - x^2)\xi^2 - 2ix\xi$ on $[-1, 1]$ ($L_x = 2.0$, $N_x = 256$, $n = 20$, $\lambda_n = n(n+1) = 420$, $L_t = 2\pi / \sqrt{\lambda_n}$, $N_t = 500$). For the degree-$n$ Legendre polynomial $P_n(x)$, the analytical eigenmode is $u(x,t) = \cos(\sqrt{\lambda_n}\,t)\,P_n(x)$, exercising nonperiodic wall clipping where $P_n(\pm 1) = (\pm 1)^n \neq 0$.

In [ ]:
import numpy as np
from scipy.special import eval_legendre
from sympy import symbols, Function, diff, Eq, I
from psiop import psiOp
from solver import PDESolver

# Symbols
t, x, xi = symbols('t x xi', real=True)
u = Function('u')

# Order of the Legendre polynomial
n = 30
lambda_n = n * (n + 1)

# ----------------------------------------------------------------------
# MATHEMATICAL FIX: 
# P_n(x) is the eigenfunction of the Legendre operator:
#   d/dx [ (1-x^2) d/dx P_n ] = -n(n+1) P_n
# In pseudo-differential symbols (where D_x <-> i*xi):
#   u_xx <-> -xi^2,  u_x <-> i*xi
# So the correct symbol for the spatial operator is:
#   -(1 - x^2)*xi^2 - 2*I*x*xi
# ----------------------------------------------------------------------
p_expr = -(1 - x**2)*xi**2 - 2*I*x*xi
equation = Eq(diff(u(t, x), t, t), psiOp(p_expr, u(t, x)))

# Creating the solver
solver = PDESolver(equation, compute_energy=False)

# Domain [-1, 1] -> Lx = 2
Lx = 2.0
Nx = 256
Lt = 2 * np.pi / np.sqrt(lambda_n)  # period of the oscillation
Nt = 500

# Initial condition: Pₙ(x) using SciPy's array-compatible evaluator
def initial_condition(x_arr):
    x_scaled = 2 * x_arr / Lx  # scales x ∈ [-1, 1]
    return eval_legendre(n, x_scaled)

def initial_velocity(x_arr):
    return np.zeros_like(x_arr)

# Exact solution
def u_exact(x_arr, t_val):
    x_scaled = 2 * x_arr / Lx
    return np.cos(np.sqrt(lambda_n) * t_val) * eval_legendre(n, x_scaled)

# Setup
solver.setup(
    Lx=Lx,
    Nx=Nx,
    Lt=Lt,
    Nt=Nt,
    # Note: P_n(±1) = (±1)^n ≠ 0, so nonperiodic is technically violated.
    # The solver will clip the edges to 0, creating a localized boundary layer.
    boundary_condition='nonperiodic',
    initial_condition=initial_condition,
    initial_velocity=initial_velocity
)

# Solving
solver.solve()

# Validation
n_test = 5
for i in range(n_test + 1):
    t_eval = i * Lt / n_test
    # We keep a relaxed threshold because the nonperiodic clipping 
    # forces a boundary layer that deviates from the true P_n(x) at the edges.
    solver.test(u_exact=u_exact, t_eval=t_eval, threshold=5.0, component='real')